# Chapter 11: Goal Setting and Monitoring

EliteAI Academy

Define measurable goals, evaluate progress, refine attempts, and monitor single- and multi-agent workflows.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL
    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 1024) -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, model=_model, max_tokens=max_tokens, system=system,
                                    messages=[{"role": "user", "content": prompt}])
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")
    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 1024) -> str:
        return _gmodel.generate_content(f"{system}\n\n{prompt}").text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply: GOAL AGENT READY"))

## 1. Core Building Blocks — Goal, Monitor, Feedback Loop

In [ ]:
@dataclass
class Goal:
    'A SMART goal definition.'
    objective:    str
    criteria:     List[str]
    max_attempts: int = 5
    timeout_secs: float = 300.0

    def prompt_criteria(self) -> str:
        return "\n".join(f"- {c}" for c in self.criteria)

@dataclass
class Attempt:
    iteration:  int
    output:     str
    score:      float
    feedback:   str
    met:        bool
    elapsed_s:  float

class GoalMonitor:
    "Evaluates an output against a goal's criteria."
    def evaluate(self, goal: Goal, output: str) -> Tuple[float, str, bool]:
        raw = call_llm(
            f"""Evaluate this output against every criterion below.

Objective: {goal.objective}

Criteria:
{goal.prompt_criteria()}

Output to evaluate:
{output}

Return ONLY valid JSON:
{{"score": <0.0-1.0>, "met": <true|false>, "feedback": "<specific issues or PASS>"}}""",
            system="You are a strict evaluator. Return only valid JSON."
        )
        try:
            data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
            return float(data["score"]), data["feedback"], bool(data["met"])
        except:
            return 0.5, "Evaluation parse failed", False

class GoalDrivenAgent:
    'Core goal-setting and monitoring loop.'
    def __init__(self, role: str, monitor: GoalMonitor):
        self.role    = role
        self.monitor = monitor
        self.history: List[Attempt] = []

    def execute(self, goal: Goal, previous: str = "", feedback: str = "") -> str:
        context = ""
        if previous:
            context = f"\n\nPrevious attempt (improve on this):\n{previous}\n\nFeedback to address:\n{feedback}"
        return call_llm(
            f"Objective: {goal.objective}\n\nSuccess criteria:\n{goal.prompt_criteria()}{context}\n\nProduce your best output.",
            system=f"You are a {self.role}. Always strive to meet every criterion."
        )

    def run(self, goal: Goal, verbose: bool = True) -> dict:
        start  = time.perf_counter()
        output = ""
        feedback = ""

        if verbose:
            print(f"\n🎯 GOAL: {goal.objective}")
            print(f"   Criteria ({len(goal.criteria)}): {goal.criteria}")
            print(f"   Max attempts: {goal.max_attempts}")

        for i in range(goal.max_attempts):
            elapsed = time.perf_counter() - start
            if elapsed > goal.timeout_secs:
                print(f"  ⏰ Timeout after {elapsed:.1f}s")
                break

            if verbose: print(f"\n[Attempt {i+1}/{goal.max_attempts}] Executing...")

            output   = self.execute(goal, output, feedback)

            score, feedback, met = self.monitor.evaluate(goal, output)
            attempt = Attempt(i+1, output, score, feedback, met, time.perf_counter()-start)
            self.history.append(attempt)

            if verbose:
                print(f"  Score: {score:.2f} | Met: {met}")
                print(f"  Feedback: {feedback[:120]}")

            if met:
                if verbose: print(f"  ✅ Goal achieved at attempt {i+1}!")
                break
            elif verbose:
                print(f"  🔄 Goal not yet met — refining...")

        best = max(self.history, key=lambda a: a.score)
        return {"final_output": output, "best_score": best.score,
                "attempts": len(self.history), "goal_met": self.history[-1].met}

    def progress_chart(self):
        print("\n📊 Progress:")
        for a in self.history:
            bar = "█" * int(a.score * 20)
            icon = "✅" if a.met else "🔄"
            print(f"  {icon} Attempt {a.attempt_id if hasattr(a,'attempt_id') else a.iteration}: [{bar:<20}] {a.score:.2f}")

print("✅ Core classes ready: Goal, GoalMonitor, GoalDrivenAgent")

## 2. Use Case 1 — Code Generation Agent (Book's Core Example)

The agent writes Python code, evaluates it against quality goals, and refines until all criteria pass.

In [ ]:
import re as _re

def clean_code_block(text: str) -> str:
    'Strip markdown code fences.'
    return _re.sub(r"```[\w]*\n?", "", text).strip()

class CodeGoalMonitor(GoalMonitor):
    'Specialised monitor for code quality evaluation.'
    def evaluate(self, goal: Goal, output: str) -> Tuple[float, str, bool]:
        code = clean_code_block(output)
        raw = call_llm(
            f"""Act as a senior Python code reviewer. Evaluate the code against ALL criteria.

Objective: {goal.objective}

Criteria (ALL must pass):
{goal.prompt_criteria()}

Code:
{code}

Return ONLY JSON: {{"score": <0.0-1.0>, "met": <true|false>, "feedback": "<specific issues or PASS>"}}""",
            system="You are a strict code reviewer. A score of 1.0 means every criterion is fully satisfied."
        )
        try:
            data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
            return float(data["score"]), data["feedback"], bool(data["met"])
        except:
            return 0.5, raw[:200], False

code_goal = Goal(
    objective="Write a Python function called `find_binary_gap(n: int) -> int` that finds the longest sequence of zeros surrounded by ones in the binary representation of a positive integer.",
    criteria=[
        "Type hints on all parameters and return value",
        "Google-style docstring with Args, Returns, Examples sections",
        "Raises ValueError for non-positive inputs",
        "Handles edge cases: n=1 (no gap), n=15 (no gap), n=9 (gap=2)",
        "Includes 3+ assert test statements at the bottom",
        "PEP 8 compliant — no lines over 88 chars",
    ],
    max_attempts=4
)

code_monitor = CodeGoalMonitor()
code_agent   = GoalDrivenAgent(role="expert Python developer", monitor=code_monitor)

print("=== CODE GENERATION GOAL AGENT ===")
result = code_agent.run(code_goal, verbose=True)

print(f"\n{'='*55}")
print(f"FINAL CODE (score={result['best_score']:.2f}, attempts={result['attempts']})")
print(f"{'='*55}")
print(clean_code_block(result["final_output"]))

## 3. Use Case 2 — Content Writing Goal Agent

In [ ]:
content_goal = Goal(
    objective="Write a LinkedIn post announcing the launch of an AI-powered project management tool called FlowAI",
    criteria=[
        "Opening hook in the first line (no 'I am excited to announce')",
        "Exactly 3 bullet points highlighting key benefits",
        "One specific statistic or data point (real or plausible)",
        "A clear call-to-action in the last line",
        "Between 150 and 250 words — count precisely",
        "At least 3 relevant hashtags at the end",
        "Professional but conversational tone — no corporate jargon",
    ],
    max_attempts=3
)

content_agent = GoalDrivenAgent(role="senior B2B content writer", monitor=GoalMonitor())

print("=== CONTENT WRITING GOAL AGENT ===")
result = content_agent.run(content_goal, verbose=True)

print(f"\n{'='*55}\nFINAL POST\n{'='*55}")
print(result["final_output"])

## 4. Use Case 3 — Customer Support Resolution Agent

Agent goal: resolve the customer issue. Monitors resolution quality, empathy, and completeness.

In [ ]:
CUSTOMER_COMPLAINT = """
I ordered a laptop stand 3 weeks ago (order #45821). The tracking says it was delivered
but I never received it. I've checked with my neighbours and building reception. 
This is the second time this has happened. I'm very frustrated and need this resolved TODAY.
"""

support_goal = Goal(
    objective=f"Resolve this customer complaint completely:\n{CUSTOMER_COMPLAINT}",
    criteria=[
        "Acknowledge frustration explicitly in the first sentence",
        "Apologise sincerely without using the word 'inconvenience'",
        "Provide a concrete resolution: replacement or full refund option",
        "Give a specific timeline (e.g. 'within 24 hours', 'by next business day')",
        "Reference the order number #45821",
        "Offer a goodwill gesture (discount, credit) due to repeat issue",
        "Provide a direct escalation contact if needed",
        "Close warmly and professionally",
    ],
    max_attempts=3
)

support_agent = GoalDrivenAgent(role="senior customer success manager", monitor=GoalMonitor())

print("=== CUSTOMER SUPPORT GOAL AGENT ===")
result = support_agent.run(support_goal, verbose=True)

print(f"\n{'='*55}\nFINAL RESPONSE (score={result['best_score']:.2f})\n{'='*55}")
print(result["final_output"])

## 5. Use Case 4 — Data Analysis Plan Agent with Multi-Dimensional Monitoring

In [ ]:
@dataclass
class MultiDimensionalGoal(Goal):
    'Goal with per-dimension scoring across multiple quality axes.'
    dimensions: Dict[str, List[str]] = field(default_factory=dict)
    weights:    Dict[str, float] = field(default_factory=dict)

class MultiDimMonitor:
    'Evaluates output across multiple weighted quality dimensions.'

    def evaluate(self, goal: MultiDimensionalGoal, output: str) -> Tuple[float, str, bool]:
        dim_scores = {}
        all_feedback = []

        for dim, criteria in goal.dimensions.items():
            criteria_text = "\n".join(f"- {c}" for c in criteria)
            raw = call_llm(
                f"""Evaluate this output on the '{dim}' dimension ONLY.

Criteria for {dim}:
{criteria_text}

Output:
{output}

Return ONLY JSON: {{"score": <0.0-1.0>, "feedback": "<issues or PASS>"}}""",
                system="You are a specialist evaluator. Return only valid JSON."
            )
            try:
                data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
                dim_scores[dim] = float(data["score"])
                all_feedback.append(f"[{dim}] {data['feedback']}")
            except:
                dim_scores[dim] = 0.5
                all_feedback.append(f"[{dim}] Parse failed")

        total_weight = sum(goal.weights.values()) or 1
        agg_score = sum(
            dim_scores.get(dim, 0) * goal.weights.get(dim, 1)
            for dim in goal.dimensions
        ) / total_weight

        print("\n  📊 Dimension scores:")
        for dim, score in dim_scores.items():
            bar = "█" * int(score * 10)
            weight = goal.weights.get(dim, 1)
            print(f"     {dim:<20} [{bar:<10}] {score:.2f} (weight={weight})")
        print(f"     {'WEIGHTED TOTAL':<20} {agg_score:.2f}")

        met = agg_score >= 0.80 and all(s >= 0.60 for s in dim_scores.values())
        return agg_score, "\n".join(all_feedback), met

analysis_goal = MultiDimensionalGoal(
    objective="Create a data analysis plan for understanding customer churn in a SaaS business",
    criteria=[],
    dimensions={
        "Completeness": [
            "Covers data sources needed (CRM, billing, usage logs)",
            "Includes feature engineering steps",
            "Specifies model selection rationale",
        ],
        "Actionability": [
            "Each step has a clear owner or tool",
            "Includes timeline estimates",
            "Ends with business recommendations",
        ],
        "Technical Rigour": [
            "Mentions validation strategy (train/test split or CV)",
            "References specific metrics (AUC-ROC, precision-recall)",
            "Addresses class imbalance if applicable",
        ],
    },
    weights={"Completeness": 0.40, "Actionability": 0.35, "Technical Rigour": 0.25},
    max_attempts=3
)

multi_monitor = MultiDimMonitor()

print("=== MULTI-DIMENSIONAL GOAL AGENT ===")
output = ""
feedback_str = ""
for i in range(analysis_goal.max_attempts):
    context = f"\nPrevious attempt:\n{output[:500]}\nFeedback:\n{feedback_str}" if output else ""
    print(f"\n[Attempt {i+1}]")
    output = call_llm(
        f"Create a data analysis plan for: {analysis_goal.objective}{context}",
        system="You are a senior data scientist creating a precise, actionable plan."
    )
    score, feedback_str, met = multi_monitor.evaluate(analysis_goal, output)
    print(f"\n  Overall: {score:.2f} | Met: {met}")
    if met:
        print("  ✅ All dimensions satisfied!")
        break

print(f"\n{'='*55}\nFINAL ANALYSIS PLAN\n{'='*55}")
print(output[:800] + "...")

## 6. Use Case 5 — Multi-Agent Goal System (Separated Concerns)

The book recommends separating producer from evaluator. Here: **Programmer Agent** writes code, **Reviewer Agent** evaluates it independently.

In [ ]:
class ProgrammerAgent:
    def write(self, task: str, feedback: str = "") -> str:
        context = f"\n\nFeedback from reviewer to address:\n{feedback}" if feedback else ""
        return call_llm(
            f"Write Python code for: {task}{context}\nReturn only the code.",
            system="You are an expert Python developer. Write clean, working, well-documented code."
        )

class ReviewerAgent:
    def review(self, task: str, code: str, criteria: List[str]) -> Tuple[float, str, bool]:
        criteria_text = "\n".join(f"- {c}" for c in criteria)
        raw = call_llm(
            f"""You are an objective code reviewer — you did NOT write this code.
Evaluate it strictly against every criterion.

Task: {task}
Criteria:
{criteria_text}

Code:
{clean_code_block(code)}

Return ONLY JSON: {{"score": <0.0-1.0>, "met": <true|false>, "feedback": "<specific issues>"}}""",
            system="You are a strict, objective code reviewer. Return only valid JSON."
        )
        try:
            data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
            return float(data["score"]), data["feedback"], bool(data["met"])
        except:
            return 0.5, raw[:200], False

class TestWriterAgent:
    def write_tests(self, task: str, code: str) -> str:
        return call_llm(
            f"Write comprehensive pytest tests for this code.\n\nTask: {task}\n\nCode:\n{clean_code_block(code)}\n\nReturn only the test code.",
            system="You are a QA engineer specialising in pytest. Write thorough, executable tests."
        )

class DocumenterAgent:
    def document(self, task: str, code: str) -> str:
        return call_llm(
            f"Write concise API documentation for this code.\n\nTask: {task}\n\nCode:\n{clean_code_block(code)}",
            system="You are a technical writer. Write clear, developer-friendly documentation."
        )

def multi_agent_goal_system(task: str, criteria: List[str], max_iterations: int = 3):
    'Full separated-concerns goal system: Programmer → Reviewer → TestWriter → Documenter'
    programmer = ProgrammerAgent()
    reviewer   = ReviewerAgent()
    tester     = TestWriterAgent()
    documenter = DocumenterAgent()

    print(f"\n{'='*60}")
    print(f"MULTI-AGENT GOAL SYSTEM")
    print(f"Task: {task[:80]}")
    print(f"{'='*60}")

    code     = ""
    feedback = ""
    score    = 0.0

    for i in range(max_iterations):
        print(f"\n[Iteration {i+1}]")

        print("  🧑‍💻 Programmer Agent writing...")
        code = programmer.write(task, feedback)

        print("  🔍 Reviewer Agent evaluating...")
        score, feedback, met = reviewer.review(task, code, criteria)
        print(f"  Score: {score:.2f} | Met: {met}")
        print(f"  Feedback: {feedback[:150]}")

        if met:
            print(f"\n  ✅ Reviewer approves code at iteration {i+1}!")
            break

    print("\n  🧪 Test Writer Agent generating tests...")
    tests = tester.write_tests(task, code)

    print("  📝 Documenter Agent writing docs...")
    docs = documenter.document(task, code)

    return {"code": clean_code_block(code), "tests": clean_code_block(tests),
            "docs": docs, "final_score": score}

result = multi_agent_goal_system(
    task="Write a Python function that counts files recursively in a directory and all subdirectories",
    criteria=[
        "Type hints on all parameters and return value",
        "Handles non-existent directory with a clear error message",
        "Uses pathlib (not os.walk)",
        "Includes a clear docstring",
        "No lines over 88 characters",
    ],
    max_iterations=3
)

print(f"\n{'='*55}\nFINAL CODE (score={result['final_score']:.2f})\n{'='*55}")
print(result["code"])
print(f"\n{'='*55}\nTESTS\n{'='*55}")
print(result["tests"][:500] + "...")
print(f"\n{'='*55}\nDOCS\n{'='*55}")
print(result["docs"][:300] + "...")

## 7. Goal Progress Dashboard

In [ ]:
def run_with_dashboard(task: str, criteria: List[str], role: str, max_attempts: int = 4):
    'Runs the goal agent and prints a rich dashboard after each attempt.'
    goal    = Goal(objective=task, criteria=criteria, max_attempts=max_attempts)
    monitor = GoalMonitor()
    agent   = GoalDrivenAgent(role=role, monitor=monitor)
    output  = ""
    feedback = ""
    scores   = []

    print(f"\n🎯 {task}")
    print(f"   {len(criteria)} criteria | max {max_attempts} attempts")

    for i in range(max_attempts):
        output   = agent.execute(goal, output, feedback)
        score, feedback, met = monitor.evaluate(goal, output)
        scores.append(score)

        bar = "█" * int(score * 20)
        icon = "✅" if met else "🔄"
        trend = ("▲" if len(scores) > 1 and scores[-1] > scores[-2] else
                 "▼" if len(scores) > 1 and scores[-1] < scores[-2] else "→")
        print(f"  {icon} Attempt {i+1}: [{bar:<20}] {score:.2f} {trend}")

        if met:
            print("  ✅ All criteria satisfied!")
            break

    improvement = scores[-1] - scores[0] if len(scores) > 1 else 0
    print(f"\n  Summary: {len(scores)} attempts | Final: {scores[-1]:.2f} | Δ {improvement:+.2f}")
    return output

print("=== GOAL AGENT DASHBOARD ===")
run_with_dashboard(
    task="Write a SQL query to find the top 5 customers by total revenue in the last 90 days",
    criteria=[
        "Uses CTEs (not subqueries) for clarity",
        "Filters last 90 days using DATEADD or equivalent",
        "Groups by customer_id AND customer_name",
        "Orders by total_revenue DESC",
        "Includes a comment explaining the query logic",
        "Handles NULL revenue values with COALESCE",
    ],
    role="senior SQL developer",
    max_attempts=3
)

## 8. SMART Goal Validator

In [ ]:
def validate_smart_goal(goal_statement: str) -> dict:
    'Evaluates whether a goal statement meets SMART criteria'
    raw = call_llm(
        f"""Evaluate this goal against SMART criteria.

Goal: \"{goal_statement}\"

SMART dimensions:
- Specific: Is it clearly defined?
- Measurable: Can progress be quantified?
- Achievable: Is it realistic?
- Relevant: Does it serve a clear purpose?
- Time-bound: Is there a timeframe?

Return ONLY JSON:
{{
  "scores": {{"specific": <0-10>, "measurable": <0-10>, "achievable": <0-10>, "relevant": <0-10>, "time_bound": <0-10>}},
  "overall": <0-10>,
  "issues": ["<list of specific problems>"],
  "improved_goal": "<rewritten SMART version>"
}}""",
        system="You are a SMART goal expert. Return only valid JSON."
    )
    data = json.loads(re.sub(r"```[\w]*", "", raw).strip())

    print(f"\n📋 SMART Evaluation: \"{goal_statement[:60]}...\"")
    print(f"\n   Dimension scores:")
    for dim, score in data["scores"].items():
        bar  = "█" * score
        icon = "✅" if score >= 7 else "⚠️ " if score >= 4 else "❌"
        print(f"   {icon} {dim.replace('_',' ').capitalize():<12} [{bar:<10}] {score}/10")
    print(f"\n   Overall: {data['overall']}/10")

    if data["issues"]:
        print(f"\n   Issues:")
        for issue in data["issues"]:
            print(f"     • {issue}")

    print(f"\n   ✨ Improved SMART goal:")
    print(f"   {data['improved_goal']}")
    return data

goals_to_test = [
    "Improve our AI agent.",
    "Reduce customer support ticket resolution time by 40% using an AI agent within Q3 2025.",
]

for g in goals_to_test:
    validate_smart_goal(g)
    print()

## 9. Real-Time Monitoring — Continuous Goal Tracking

In [ ]:
import asyncio

async def acall_llm(prompt: str, system: str = "You are a helpful assistant.") -> str:
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, lambda: call_llm(prompt, system))

@dataclass
class MonitoredTask:
    name:    str
    goal:    str
    metrics: Dict[str, float] = field(default_factory=dict)
    status:  str = "pending"

class RealTimeGoalMonitor:
    'Monitors multiple parallel tasks toward a shared objective.'
    def __init__(self, project_goal: str, kpi_thresholds: Dict[str, float]):
        self.project_goal  = project_goal
        self.thresholds    = kpi_thresholds
        self.tasks:        List[MonitoredTask] = []
        self.alerts:       List[str] = []

    def add_task(self, task: MonitoredTask):
        self.tasks.append(task)

    async def check_task(self, task: MonitoredTask) -> dict:
        'Simulate executing a task and monitoring its KPIs.'

        result_text = await acall_llm(
            f"Perform this task and report KPIs: {task.goal}",
            system="You are an agent executing a task. Be concise."
        )

        kpi_raw = await acall_llm(
            f"""From this task result, estimate these KPIs as 0.0-1.0 scores.
Task: {task.goal}\nResult: {result_text}
KPIs to score: {list(self.thresholds.keys())}
Return ONLY JSON: {{"kpi_name": score, ...}}""",
            system="You are a KPI extractor. Return only valid JSON."
        )
        try:
            kpis = json.loads(re.sub(r"```[\w]*", "", kpi_raw).strip())
        except:
            kpis = {k: 0.5 for k in self.thresholds}

        task.metrics = kpis

        breaches = [
            f"{k}={v:.2f} (threshold={self.thresholds[k]:.2f})"
            for k, v in kpis.items()
            if k in self.thresholds and v < self.thresholds[k]
        ]

        if breaches:
            alert = f"⚠️  [{task.name}] KPI breach: {', '.join(breaches)}"
            self.alerts.append(alert)
            task.status = "failed"
        else:
            task.status = "complete"

        return {"task": task.name, "kpis": kpis, "status": task.status,
                "breaches": breaches}

    async def monitor_all(self):
        'Monitor all tasks in parallel.'
        print(f"\n🎯 Project goal: {self.project_goal}")
        print(f"   Monitoring {len(self.tasks)} tasks in parallel...\n")

        results = await asyncio.gather(*[self.check_task(t) for t in self.tasks])

        for r in results:
            icon = "✅" if r["status"] == "complete" else "❌"
            print(f"  {icon} {r['task']}: {r['status']}")
            for kpi, score in r["kpis"].items():
                threshold = self.thresholds.get(kpi, 0)
                ok = "✓" if score >= threshold else "✗"
                print(f"     {ok} {kpi}: {score:.2f} (min={threshold:.2f})")

        if self.alerts:
            print(f"\n🚨 ALERTS:")
            for a in self.alerts:
                print(f"  {a}")
        else:
            print("\n✅ All tasks met KPI thresholds!")

        success_rate = sum(1 for t in self.tasks if t.status == "complete") / len(self.tasks)
        print(f"\n📊 Project success rate: {success_rate:.0%}")

monitor = RealTimeGoalMonitor(
    project_goal="Release FlowAI v2.0 with zero critical bugs by end of sprint",
    kpi_thresholds={"completeness": 0.80, "quality": 0.75, "speed": 0.60}
)

monitor.add_task(MonitoredTask("Backend API",     "Implement REST API endpoints for user authentication"))
monitor.add_task(MonitoredTask("Frontend UI",     "Build React components for the dashboard"))
monitor.add_task(MonitoredTask("Test Suite",      "Write integration tests for all new endpoints"))
monitor.add_task(MonitoredTask("Documentation",  "Update API docs and changelog"))

await monitor.monitor_all()

## 10. Quick Reference — Pattern Rules

In [ ]:
rules = """
╔══════════════════════════════════════════════════════════════════╗
║  GOAL SETTING & MONITORING — QUICK REFERENCE                     ║
╠══════════════════════════════════════════════════════════════════╣
║  GOAL DEFINITION               ║  MONITORING                     ║
║  • Use SMART criteria          ║  • Evaluate after every attempt ║
║  • Be specific and measurable  ║  • Score 0.0–1.0 per criterion  ║
║  • Cap max_attempts (3–5)      ║  • Track trend (▲▼→)            ║
║  • Set a timeout_secs          ║  • Alert on KPI breaches        ║
╠══════════════════════════════════════════════════════════════════╣
║  PRODUCER vs EVALUATOR                                           ║
║  ✅ Separate agents = more objective evaluation                  ║
║  ⚠️  Same LLM for both = cognitive bias risk                     ║
║  → Use different system prompts at minimum                       ║
╠══════════════════════════════════════════════════════════════════╣
║  STOPPING CONDITIONS (use at least ONE)                          ║
║  • All criteria return met=True                                  ║
║  • Score >= threshold (e.g. 0.85)                                ║
║  • Max iterations reached                                        ║
║  • Timeout elapsed                                               ║
╠══════════════════════════════════════════════════════════════════╣
║  CAVEATS                                                         ║
║  • LLM may hallucinate goal satisfaction — always test outputs   ║
║  • No infinite loops — always set max_attempts                   ║
║  • Cost = attempts × 2 LLM calls (execute + evaluate)           ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(rules)

## 11. Exercises

- **Exercise 1 — Timeout Guard:** Modify `GoalDrivenAgent.run` to enforce `goal.timeout_secs` strictly using `time.perf_counter()`. If a single `execute()` call exceeds half the timeout, skip the monitor step and go straight to the next iteration.

- **Exercise 2 — Criterion-Level Scoring:** Extend `GoalMonitor.evaluate` to return a score per criterion (not just aggregate). Show a per-criterion pass/fail table in the dashboard.

- **Exercise 3 — Escalation Policy:** Add an `escalation_fn` callback to `GoalDrivenAgent`. If after 2 attempts the score hasn't improved by at least 0.05, call the escalation function (e.g. switch to a more powerful model or route to a human).

- **Exercise 4 — Goal Decomposition:** Build a `GoalDecomposer` that takes a high-level goal (e.g. "Launch a new product") and automatically generates 3–5 SMART sub-goals using an LLM. Then run a `GoalDrivenAgent` on each sub-goal in parallel.

- **Exercise 5 — LangChain LCEL Integration:** Rewrite `GoalDrivenAgent` as a LangChain `RunnableWithFallbacks` chain where the execute step and evaluate step are LCEL chains connected by a conditional router.

**Next:** Chapter 12 — Human-in-the-Loop